# Funding Rate Risk Monitoring

This notebook analyzes perpetual futures funding rates to monitor basis risk, detect regime changes, and estimate funding costs. A practical workflow for risk analysts and portfolio managers managing perpetual futures positions.

**What you'll learn:**
- Analyzing 8-hourly funding rate time series
- Using computed features for trend and volatility monitoring
- Mark-index basis analysis
- Funding rate regime detection
- Annualized funding cost estimation
- Aggregated risk metrics with DuckDB SQL

**Dataset:** `funding_rates` (180 rows, 60 days of 8-hourly BTCUSDT funding snapshots) + `funding_factors`

## 1. Pipeline Setup

In [ ]:
import subprocess
import os
from pathlib import Path

# Navigate to the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif not Path("config/datasets.yaml").exists():
    root = Path.cwd()
    while not (root / "config" / "datasets.yaml").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

# Initialize platform and run the funding rates pipeline
subprocess.run(["adp", "init"], capture_output=True, check=True)
subprocess.run(["adp", "ingest", "funding_rates", "--force"], capture_output=True, check=True)
subprocess.run(["adp", "snapshot", "create", "funding_rates"], capture_output=True, check=True)
# funding_factors: SMA, EWMA, and mark price realized volatility
subprocess.run(["adp", "features", "build", "funding_rates", "funding_factors"], capture_output=True, check=True)
print("Pipeline complete: funding_rates -> funding_factors")

In [ ]:
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from adp import load_dataset, load_features, query_dataset, query_features

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

## 2. Funding Rate Overview

Perpetual futures funding rates are periodic payments between long and short holders, settled every 8 hours. Positive rates mean longs pay shorts (market is bullish); negative rates mean shorts pay longs.

In [ ]:
df = load_dataset("funding_rates").collect().sort("timestamp")

print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
num_days = max((df["timestamp"].max() - df["timestamp"].min()).days, 1)
print(f"Funding snapshots per day: {df.shape[0] / num_days:.0f} (every 8 hours)")

print("\n=== Funding Rate Statistics ===")
fr = df["funding_rate"]
print(f"  Mean:   {fr.mean():.8f}")
print(f"  Median: {fr.median():.8f}")
print(f"  Min:    {fr.min():.8f}")
print(f"  Max:    {fr.max():.8f}")
print(f"  Std:    {fr.std():.8f}")

print("\nFirst 5 rows:")
print(df.head(5))

## 3. Funding Rate Time Series

Visualize the raw funding rate over the 60-day period with mean and standard deviation bands.

In [ ]:
ts = df["timestamp"].to_list()
rates = df["funding_rate"].to_list()
mean_rate = df["funding_rate"].mean()
std_rate = df["funding_rate"].std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts, rates, linewidth=1, alpha=0.8, label="Funding Rate")
ax.axhline(y=mean_rate, color="red", linestyle="--", linewidth=1, label=f"Mean ({mean_rate:.6f})")
ax.axhline(y=mean_rate + std_rate, color="orange", linestyle=":", linewidth=1, label=f"+1 Std")
ax.axhline(y=mean_rate - std_rate, color="orange", linestyle=":", linewidth=1, label=f"-1 Std")
ax.axhline(y=0, color="gray", linewidth=0.5)
ax.set_xlabel("Date")
ax.set_ylabel("Funding Rate")
ax.set_title("BTCUSDT Perpetual Funding Rate (8-hourly)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Analysis: Trend & Volatility

Load the `funding_factors` features and overlay the smoothed trend indicators on the raw rate.

In [ ]:
feat_df = load_features("funding_rates", "funding_factors").collect().sort("timestamp")

print(f"Shape: {feat_df.shape}")
print(f"Columns: {feat_df.columns}")

ts = feat_df["timestamp"].to_list()
# Convert nulls to NaN for consistent matplotlib line-break behavior
plot_feat = feat_df.fill_null(float("nan"))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Subplot 1: Raw funding rate overlaid with smoothed trend indicators ---
axes[0].plot(ts, plot_feat["funding_rate"].to_list(), alpha=0.4, linewidth=0.8, label="Raw Rate")
axes[0].plot(ts, plot_feat["funding_sma_10"].to_list(), linewidth=1.5, label="SMA(10)")
axes[0].plot(ts, plot_feat["funding_ewma_20"].to_list(), linewidth=1.5, linestyle="--", label="EWMA(20)")
axes[0].set_ylabel("Funding Rate")
axes[0].set_title("Funding Rate with Trend Indicators")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# --- Subplot 2: Mark price realized volatility (rolling 10-period std of returns) ---
axes[1].plot(ts, plot_feat["mark_realized_vol_10"].to_list(), color="purple", linewidth=1.2)
axes[1].set_ylabel("Realized Volatility")
axes[1].set_title("Mark Price Realized Volatility (10-period)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Mark-Index Basis Analysis

The basis (mark price - index price) reflects the premium or discount of the perpetual contract relative to spot. Persistent positive basis indicates bullish sentiment; persistent negative basis indicates bearish sentiment.

In [ ]:
# Compute absolute basis (USD) and relative basis (bps) between mark and index price
basis_df = df.with_columns(
    (pl.col("mark_price") - pl.col("index_price")).alias("basis"),
    ((pl.col("mark_price") - pl.col("index_price")) / pl.col("index_price") * 10000)
    .round(2)
    .alias("basis_bps"),
)

print("=== Basis Statistics ===")
basis = basis_df["basis"]
basis_bps = basis_df["basis_bps"]
print(f"  Mean basis:     ${basis.mean():.2f} ({basis_bps.mean():.1f} bps)")
print(f"  Std basis:      ${basis.std():.2f} ({basis_bps.std():.1f} bps)")
print(f"  Min basis:      ${basis.min():.2f} ({basis_bps.min():.1f} bps)")
print(f"  Max basis:      ${basis.max():.2f} ({basis_bps.max():.1f} bps)")

# Color-coded bar chart: green for positive basis (premium), red for negative (discount)
fig, ax = plt.subplots(figsize=(14, 4))
basis_vals = basis_df["basis_bps"].to_list()
colors = ["green" if b >= 0 else "red" for b in basis_vals]
ax.bar(range(len(basis_vals)), basis_vals, color=colors, alpha=0.7, width=1.0)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_ylabel("Basis (bps)")
ax.set_title("Mark-Index Basis Over Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Funding Rate Regime Detection

Classify each funding snapshot into regimes based on the rate level:
- **Positive**: funding_rate > 0.01% (longs pay shorts, bullish sentiment)
- **Neutral**: -0.01% <= funding_rate <= 0.01%
- **Negative**: funding_rate < -0.01% (shorts pay longs, bearish sentiment)

In [ ]:
# Binance default funding rate is 0.01% (1 bps). Use as the neutral band boundary.
NEUTRAL_BAND = 0.0001  # 0.01% = 1 basis point

# Classify each snapshot into a funding regime
regime_df = df.with_columns(
    pl.when(pl.col("funding_rate") > NEUTRAL_BAND)
    .then(pl.lit("Positive"))
    .when(pl.col("funding_rate") < -NEUTRAL_BAND)
    .then(pl.lit("Negative"))
    .otherwise(pl.lit("Neutral"))
    .alias("regime")
)

# Regime summary: average rate, volatility, and time proportion
regime_summary = (
    regime_df
    .group_by("regime")
    .agg([
        pl.col("timestamp").count().alias("snapshots"),
        pl.col("funding_rate").mean().round(8).alias("avg_rate"),
        pl.col("funding_rate").std().round(8).alias("std_rate"),
        pl.col("mark_price").mean().round(2).alias("avg_mark_price"),
    ])
    .with_columns(
        (pl.col("snapshots") / regime_df.shape[0] * 100)
        .round(1)
        .alias("pct_of_time")
    )
    .sort("regime")
)

print("=== Funding Rate Regimes ===")
print(regime_summary)

# Count regime transitions (consecutive snapshots with different regime labels)
regimes = regime_df["regime"].to_list()
transitions = sum(1 for i in range(1, len(regimes)) if regimes[i] != regimes[i - 1])
print(f"\nRegime transitions: {transitions} over {df.shape[0]} snapshots")
print(f"Average regime duration: {df.shape[0] / max(transitions, 1):.1f} snapshots")

## 7. Annualized Funding Cost

Estimate the annualized cost of holding a perpetual futures position. With 3 funding payments per day (every 8 hours), the simple (non-compounded) annualized rate is:

$$\text{annualized} = \text{funding\_rate} \times 3 \times 365$$

> **Note:** This is the industry-standard linear annualization. For large funding rates, the compounded formula `((1 + rate)^1095 - 1)` is more accurate, but the difference is negligible at typical rates (~0.01% per 8h).

In [ ]:
# Annualize: 3 funding payments/day * 365 days, expressed as percentage
cost_df = df.with_columns(
    (pl.col("funding_rate") * 3 * 365 * 100).round(4).alias("annualized_pct"),
)

# Cumulative funding cost: running sum of per-period rates (additive approximation)
cost_df = cost_df.with_columns(
    (pl.col("funding_rate").cum_sum() * 100).alias("cumulative_pct"),
)

print("=== Annualized Funding Cost ===")
ann = cost_df["annualized_pct"]
print(f"  Current (latest):  {ann.to_list()[-1]:.2f}%")
print(f"  Average:           {ann.mean():.2f}%")
print(f"  Min:               {ann.min():.2f}%")
print(f"  Max:               {ann.max():.2f}%")

cum_pct = cost_df["cumulative_pct"].to_list()[-1]
print(f"\n  Cumulative cost ({num_days} days, long position): {cum_pct:.4f}%")

ts = cost_df["timestamp"].to_list()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# --- Subplot 1: Per-snapshot annualized funding rate ---
axes[0].plot(ts, cost_df["annualized_pct"].to_list(), linewidth=1.2)
axes[0].axhline(y=0, color="gray", linewidth=0.5)
axes[0].set_ylabel("Annualized Rate (%)")
axes[0].set_title("Annualized Funding Cost")
axes[0].grid(True, alpha=0.3)

# --- Subplot 2: Cumulative funding cost for a long position ---
cum_pcts = cost_df["cumulative_pct"].to_list()
axes[1].fill_between(ts, cum_pcts, alpha=0.3)
axes[1].plot(ts, cum_pcts, linewidth=1.5)
axes[1].set_ylabel("Cumulative Funding Cost (%)")
axes[1].set_title("Cumulative Funding Cost (Long Position)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. DuckDB: Aggregated Risk Metrics

Use SQL to compute daily summary statistics for risk reporting.

In [ ]:
daily_risk = query_dataset(
    "funding_rates",
    """
    SELECT
        CAST(timestamp AS DATE) AS date,
        COUNT(*) AS snapshots,
        ROUND(AVG(funding_rate) * 1e4, 4) AS avg_rate_bps,
        ROUND(MAX(ABS(funding_rate)) * 1e4, 4) AS max_abs_rate_bps,
        ROUND(AVG(mark_price - index_price), 2) AS avg_basis_usd,
        ROUND(AVG(mark_price), 2) AS avg_mark_price
    FROM dataset
    GROUP BY CAST(timestamp AS DATE)
    ORDER BY date
    """,
)

print("=== Daily Funding Risk Summary ===")
print(daily_risk)

In [ ]:
# Feature-based risk query: identify high-volatility funding periods
high_vol = query_features(
    "funding_rates",
    "funding_factors",
    """
    SELECT
        timestamp,
        funding_rate,
        funding_sma_10,
        funding_ewma_20,
        mark_realized_vol_10
    FROM features
    WHERE mark_realized_vol_10 IS NOT NULL
    ORDER BY mark_realized_vol_10 DESC
    LIMIT 10
    """,
)

print("=== Top 10 Highest Mark Price Volatility Periods ===")
print(high_vol)

## Key Findings

This notebook demonstrates a risk monitoring workflow for perpetual futures funding rates:

1. **Funding Rate Trends** — SMA and EWMA smoothing reveals the underlying trend beneath noisy 8-hourly rates
2. **Mark-Index Basis** — The premium/discount of perpetual vs spot reveals market sentiment and arbitrage opportunities
3. **Regime Detection** — Classifying rates into positive/neutral/negative regimes helps identify sentiment shifts
4. **Funding Cost** — Annualized cost estimation is critical for position sizing and carry trade analysis
5. **Risk Metrics** — Daily aggregated metrics provide a dashboard-ready summary for risk reporting

These analyses are essential for:
- **Portfolio managers** — estimating funding costs for perpetual futures positions
- **Risk analysts** — monitoring basis risk and funding rate volatility
- **Traders** — timing entries/exits based on funding rate regimes